# Visualizing Table Lookups: Full Table Scan vs. B-Tree Index

This notebook demonstrates what happens when a database searches for a row:

1. **Without an index:** the database may scan the table looking for matching rows.
2. **With a B-tree index:** the database can use the index to narrow the search quickly.

The notebook has two parts:

- A **small visual example** so you can see the search process.
- A **real SQLite example** using `EXPLAIN QUERY PLAN` and timing tests.

> The B-tree diagrams are simplified teaching models. Actual database index pages contain many more keys and pointers than shown here.

## Learning objectives

By the end of the notebook, you should be able to:

- Explain why a lookup without an index may require a full table scan.
- Explain how a B-tree index reduces the amount of data the database must inspect.
- Use `EXPLAIN QUERY PLAN` to see whether SQLite is scanning a table or using an index.
- Compare lookup performance before and after creating an index.

In [ ]:
import math
import random
import sqlite3
import statistics
import time

import matplotlib.pyplot as plt
import pandas as pd

random.seed(523)

# Part 1: A small table we can visualize

The table below is stored in physical row order. The `lookup_key` values are deliberately not sorted.

We will search for a value in `lookup_key`, not `row_id`.

In [ ]:
toy_keys = [
    18, 4, 29, 11, 2, 31, 7, 23,
    15, 1, 26, 9, 20, 13, 32, 6,
    24, 17, 3, 28, 12, 22, 8, 30,
    5, 16, 27, 10, 21, 14, 25, 19
]

toy = pd.DataFrame({
    "row_id": range(1, len(toy_keys) + 1),
    "lookup_key": toy_keys,
    "payload": [f"row_{i:02d}" for i in range(1, len(toy_keys) + 1)]
})

toy

## Lookup without an index

Without an index on `lookup_key`, the database does not have a separate ordered structure telling it where each value is located.

For a query such as:

```sql
SELECT *
FROM demo
WHERE lookup_key = 27;
```

SQLite may perform a **table scan**.

The visualization below shows the physical rows inspected as the scan moves through the table.

In [ ]:
def visualize_table_scan(df, target):
    positions = df.index[df["lookup_key"] == target].tolist()

    if not positions:
        raise ValueError(f"{target} is not in the table.")

    # Teaching view: stop at the first matching row.
    first_match = positions[0]
    inspected = first_match + 1

    fig, ax = plt.subplots(figsize=(12, 5))

    x = list(range(len(df)))
    bars = ax.bar(x, [1] * len(df))

    for i, bar in enumerate(bars):
        if i > first_match:
            bar.set_alpha(0.25)
        if i == first_match:
            bar.set_hatch("//")

    ax.set_xticks(x)
    ax.set_xticklabels(df["lookup_key"], fontsize=8)
    ax.set_yticks([])
    ax.set_xlabel("Rows in physical table order")
    ax.set_title(
        f"Table scan looking for lookup_key = {target}\n"
        f"First match reached after inspecting {inspected} of {len(df)} rows"
    )

    plt.show()

    print(f"Target: {target}")
    print(f"First matching physical row: {first_match + 1}")
    print(f"Rows inspected to reach first match: {inspected}")

visualize_table_scan(toy, 27)

### Important detail

The picture stops at the first matching row to make the idea easy to see.

However, if `lookup_key` is **not guaranteed to be unique**, a database may need to continue scanning the remaining rows to find additional matches. A full scan can therefore involve reading the entire table.

# Part 2: What the B-tree index adds

A B-tree index creates a separate, ordered structure based on the indexed column.

Conceptually, instead of checking physical table rows one at a time, the database can:

1. Read an index page.
2. Decide which branch can contain the key.
3. Follow that branch.
4. Continue until it reaches the index entry.
5. Use the stored row reference to find the table row if additional columns are needed.

The next diagram shows a simplified multi-level index.

In [ ]:
def draw_simplified_btree(target=27):
    if not 1 <= target <= 32:
        raise ValueError("Choose a target from 1 through 32.")

    leaf_ranges = [(1, 8), (9, 16), (17, 24), (25, 32)]
    root_labels = ["1 to 8", "9 to 16", "17 to 24", "25 to 32"]

    selected_leaf = next(
        i for i, (low, high) in enumerate(leaf_ranges)
        if low <= target <= high
    )

    fig, ax = plt.subplots(figsize=(12, 6))
    ax.axis("off")

    root_x = [0.15, 0.38, 0.62, 0.85]
    root_y = 0.78
    leaf_y = 0.35

    ax.text(
        0.5, 0.94,
        f"Simplified B-tree lookup for lookup_key = {target}",
        ha="center", va="center", fontsize=14, transform=ax.transAxes
    )

    ax.text(
        0.5, 0.84,
        "Root or internal index page",
        ha="center", va="center", fontsize=11, transform=ax.transAxes
    )

    for i, (x, label) in enumerate(zip(root_x, root_labels)):
        alpha = 1.0 if i == selected_leaf else 0.3

        ax.text(
            x, root_y, label,
            ha="center", va="center",
            bbox=dict(boxstyle="round,pad=0.5", alpha=alpha),
            transform=ax.transAxes
        )

        ax.plot(
            [x, x],
            [root_y - 0.05, leaf_y + 0.11],
            alpha=alpha,
            transform=ax.transAxes
        )

        low, high = leaf_ranges[i]
        leaf_text = "  ".join(str(v) for v in range(low, high + 1))

        ax.text(
            x, leaf_y, leaf_text,
            ha="center", va="center", fontsize=10,
            bbox=dict(boxstyle="round,pad=0.6", alpha=alpha),
            transform=ax.transAxes
        )

    ax.text(
        0.5, 0.14,
        f"The root directs the search to the leaf page containing {target}.",
        ha="center", va="center", fontsize=11, transform=ax.transAxes
    )

    ax.text(
        0.5, 0.07,
        "The database does not need to inspect every table row to locate the key.",
        ha="center", va="center", fontsize=10, transform=ax.transAxes
    )

    plt.show()

draw_simplified_btree(27)

## Compare the amount of search work

For a table scan, the number of rows examined can grow roughly with the size of the table.

For a balanced tree, the number of levels grows much more slowly.

The chart below is illustrative. It compares:

- `N` steps for a worst-case table scan
- `log2(N)` as a simple approximation of tree-search growth

Real B-tree indexes are **multiway trees**, so a database page can hold many keys and the real tree height is often even smaller than a binary-tree approximation suggests.

In [ ]:
sizes = [10, 100, 1_000, 10_000, 100_000, 1_000_000]

comparison = pd.DataFrame({
    "rows": sizes,
    "table_scan_steps": sizes,
    "log2_index_steps": [math.ceil(math.log2(n)) for n in sizes]
})

comparison

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

x = range(len(comparison))
width = 0.38

ax.bar(
    [i - width/2 for i in x],
    comparison["table_scan_steps"],
    width=width,
    label="Worst-case table scan"
)

ax.bar(
    [i + width/2 for i in x],
    comparison["log2_index_steps"],
    width=width,
    label="Approx. tree search"
)

ax.set_yscale("log")
ax.set_xticks(list(x))
ax.set_xticklabels([f"{n:,}" for n in comparison["rows"]])
ax.set_xlabel("Number of table rows")
ax.set_ylabel("Approximate search work, log scale")
ax.set_title("How search work grows")
ax.legend()

plt.show()

# Part 3: Real SQLite demonstration

Now we will create a real SQLite table in memory.

The table has:

- `row_id` as its primary key
- `lookup_key` as the column we will search
- `payload` as example data

We deliberately **do not** index `lookup_key` at first.

In [ ]:
N = 200_000

conn = sqlite3.connect(":memory:")
cur = conn.cursor()

cur.execute("""
CREATE TABLE demo (
    row_id INTEGER PRIMARY KEY,
    lookup_key INTEGER NOT NULL,
    payload TEXT NOT NULL
)
""")

keys = list(range(1, N + 1))
random.shuffle(keys)

rows = [
    (row_id, lookup_key, f"payload for row {row_id}")
    for row_id, lookup_key in enumerate(keys, start=1)
]

cur.executemany(
    "INSERT INTO demo (row_id, lookup_key, payload) VALUES (?, ?, ?)",
    rows
)

conn.commit()

print(f"Created demo table with {N:,} rows.")

## Check the query plan before creating the index

SQLite's `EXPLAIN QUERY PLAN` tells us how it plans to execute the query.

Look for the word **SCAN**.

In [ ]:
target = keys[-10]

query = """
SELECT row_id, lookup_key, payload
FROM demo
WHERE lookup_key = ?
"""

plan_without_index = pd.read_sql_query(
    "EXPLAIN QUERY PLAN " + query,
    conn,
    params=(target,)
)

print(f"Searching for lookup_key = {target}")
plan_without_index

If SQLite reports something similar to:

```text
SCAN demo
```

it is scanning the table because there is no useful index on `lookup_key`.

In [ ]:
def time_lookup(connection, sql, parameter, repeats=20):
    times_ms = []

    for _ in range(repeats):
        start = time.perf_counter()
        connection.execute(sql, (parameter,)).fetchall()
        end = time.perf_counter()

        times_ms.append((end - start) * 1000)

    return {
        "runs": repeats,
        "median_ms": statistics.median(times_ms),
        "mean_ms": statistics.mean(times_ms),
        "min_ms": min(times_ms),
        "max_ms": max(times_ms),
    }

before_timing = time_lookup(conn, query, target)

pd.DataFrame([before_timing])

# Part 4: Create a B-tree index

SQLite uses a B-tree structure for ordinary indexes.

We will create an index on the column used in the `WHERE` clause:

```sql
CREATE INDEX idx_demo_lookup_key
ON demo(lookup_key);
```

In [ ]:
cur.execute(
    "CREATE INDEX idx_demo_lookup_key ON demo(lookup_key)"
)

conn.commit()

print("Index created.")

## Inspect the index definition

In [ ]:
index_list = pd.read_sql_query(
    "PRAGMA index_list('demo')",
    conn
)

index_info = pd.read_sql_query(
    "PRAGMA index_info('idx_demo_lookup_key')",
    conn
)

display(index_list)
display(index_info)

## Check the query plan again

Run the same lookup after creating the index.

This time, look for **SEARCH** and the index name.

In [ ]:
plan_with_index = pd.read_sql_query(
    "EXPLAIN QUERY PLAN " + query,
    conn,
    params=(target,)
)

plan_with_index

A typical indexed plan will look similar to:

```text
SEARCH demo USING INDEX idx_demo_lookup_key (lookup_key=?)
```

That means SQLite is using the B-tree index instead of scanning the entire table.

In [ ]:
after_timing = time_lookup(conn, query, target)

timing_comparison = pd.DataFrame([
    {
        "lookup_method": "Without index",
        **before_timing
    },
    {
        "lookup_method": "With B-tree index",
        **after_timing
    }
])

timing_comparison

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ax.bar(
    timing_comparison["lookup_method"],
    timing_comparison["median_ms"]
)

ax.set_ylabel("Median lookup time (milliseconds)")
ax.set_title("Same lookup before and after creating an index")

for i, value in enumerate(timing_comparison["median_ms"]):
    ax.text(i, value, f"{value:.3f} ms", ha="center", va="bottom")

plt.show()

## What changed?

The SQL query did **not** change.

Only the database structure changed:

```sql
CREATE INDEX idx_demo_lookup_key
ON demo(lookup_key);
```

Before the index, SQLite had no ordered lookup structure for `lookup_key`.

After the index, SQLite could use the B-tree to find the matching key much more directly.

This is one reason indexes can dramatically improve lookup performance on large tables.

# Part 5: See indexed values in order

An index maintains its keys in sorted order.

The SQL below forces SQLite to use the index while returning the first few indexed values.

In [ ]:
indexed_values = pd.read_sql_query("""
SELECT lookup_key, row_id
FROM demo INDEXED BY idx_demo_lookup_key
ORDER BY lookup_key
LIMIT 25
""", conn)

indexed_values

Notice that the `lookup_key` values are returned in index order even though those values were inserted into the table in random physical order.

The index does **not** rearrange the table itself. It creates a separate ordered structure that points back to rows.

# Part 6: Student experiments

Try each experiment and explain what happens.

### Experiment A: Search for a different value

Change:

```python
target = keys[-10]
```

to another value between `1` and `N`.

Does the query plan change?

### Experiment B: Remove the index

Run:

```sql
DROP INDEX idx_demo_lookup_key;
```

Then run `EXPLAIN QUERY PLAN` again.

What happens?

### Experiment C: Create an index on a different column

Try:

```sql
CREATE INDEX idx_demo_payload
ON demo(payload);
```

Then compare a lookup using `lookup_key` with one using `payload`.

Does SQLite use an index only because an index exists somewhere on the table?

### Experiment D: Increase the table size

Try:

```python
N = 1_000_000
```

Recreate the database and compare the lookup timings again.

How does increasing table size affect the scan compared with the indexed lookup?

# Key takeaway

A table scan asks, in effect:

> "Check rows until you determine which rows match."

A B-tree index asks:

> "Use the ordered index structure to narrow down where the matching value is located."

For a very small table, the difference may not matter much. As tables grow, indexes become increasingly important for columns that are frequently used for lookups, joins, sorting, and filtering.

Indexes are not free. They require storage and must also be updated when rows are inserted, deleted, or indexed values are changed.